In [1]:
import pandas as pd, numpy as np, glob, os, json
pd.set_option("display.max_columns", 100)

RAW = "data/raw"
STUDIES = sorted([d for d in os.listdir(RAW) if d.endswith("pan_can_atlas_2018")])
print(STUDIES)

def load_clinical(study):
    p = os.path.join(RAW, study, "data_clinical_patient.txt")
    df = pd.read_csv(p, sep="\t", comment="#", low_memory=False)
    df["STUDY"] = study.split("_")[0].upper()
    return df

clin = pd.concat([load_clinical(s) for s in STUDIES], ignore_index=True)
print(clin.shape)
print(sorted(clin.columns.tolist()))
clin.head(3)

['blca_tcga_pan_can_atlas_2018', 'brca_tcga_pan_can_atlas_2018', 'cesc_tcga_pan_can_atlas_2018', 'coadread_tcga_pan_can_atlas_2018', 'hnsc_tcga_pan_can_atlas_2018', 'lihc_tcga_pan_can_atlas_2018', 'luad_tcga_pan_can_atlas_2018', 'lusc_tcga_pan_can_atlas_2018', 'ov_tcga_pan_can_atlas_2018', 'stad_tcga_pan_can_atlas_2018']
(5359, 39)
['AGE', 'AJCC_PATHOLOGIC_TUMOR_STAGE', 'AJCC_STAGING_EDITION', 'CANCER_TYPE_ACRONYM', 'DAYS_LAST_FOLLOWUP', 'DAYS_TO_BIRTH', 'DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS', 'DFS_MONTHS', 'DFS_STATUS', 'DSS_MONTHS', 'DSS_STATUS', 'ETHNICITY', 'FORM_COMPLETION_DATE', 'GENETIC_ANCESTRY_LABEL', 'HISTORY_NEOADJUVANT_TRTYN', 'ICD_10', 'ICD_O_3_HISTOLOGY', 'ICD_O_3_SITE', 'INFORMED_CONSENT_VERIFIED', 'IN_PANCANPATHWAYS_FREEZE', 'NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT', 'OS_MONTHS', 'OS_STATUS', 'OTHER_PATIENT_ID', 'PATH_M_STAGE', 'PATH_N_STAGE', 'PATH_T_STAGE', 'PATIENT_ID', 'PERSON_NEOPLASM_CANCER_STATUS', 'PFS_MONTHS', 'PFS_STATUS', 'PRIMARY_LYMPH_NODE_PRESENTATION_ASSE

,PATIENT_ID,SUBTYPE,CANCER_TYPE_ACRONYM,OTHER_PATIENT_ID,AGE,SEX,AJCC_PATHOLOGIC_TUMOR_STAGE,AJCC_STAGING_EDITION,DAYS_LAST_FOLLOWUP,DAYS_TO_BIRTH,DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS,ETHNICITY,FORM_COMPLETION_DATE,HISTORY_NEOADJUVANT_TRTYN,ICD_10,ICD_O_3_HISTOLOGY,ICD_O_3_SITE,INFORMED_CONSENT_VERIFIED,NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT,PATH_M_STAGE,PATH_N_STAGE,PATH_T_STAGE,PERSON_NEOPLASM_CANCER_STATUS,PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT,PRIOR_DX,RACE,RADIATION_THERAPY,WEIGHT,IN_PANCANPATHWAYS_FREEZE,OS_STATUS,OS_MONTHS,DSS_STATUS,DSS_MONTHS,DFS_STATUS,DFS_MONTHS,PFS_STATUS,PFS_MONTHS,GENETIC_ANCESTRY_LABEL,STUDY
0,TCGA-2F-A9KO,BLCA,BLCA,DD3A6357-9087-44B2-9956-F981E0DE6F1C,63.0,Male,STAGE IV,6TH,678.0,-23323.0,0.0,Not Hispanic Or Latino,3/19/14,No,C67.4,8120/3,C67.4,Yes,Yes,M0,N1,T3,With Tumor,Yes,No,White,No,65.0,Yes,1:DECEASED,24.131242,1:DEAD WITH TUMOR,24.131242,NaN,NaN,1:PROGRESSION,21.402505,EUR,BLCA
1,TCGA-2F-A9KP,BLCA,BLCA,65D1EAEC-28DB-4A41-A4DB-1A710FCB24AD,66.0,Male,STAGE IV,7TH,NaN,-24428.0,0.0,Not Hispanic Or Latino,3/19/14,No,C67.2,8120/3,C67.2,Yes,Yes,MX,N2,T3A,With Tumor,Yes,No,White,No,130.0,Yes,1:DECEASED,11.966992,1:DEAD WITH TUMOR,11.966992,NaN,NaN,1:PROGRESSION,11.276589,EUR,BLCA
2,TCGA-2F-A9KQ,BLCA,BLCA,25EAF3F9-C364-423C-AAAE-925E7B393AFC,69.0,Male,STAGE III,6TH,2886.0,-25259.0,0.0,Not Hispanic Or Latino,3/19/14,No,C67.9,8120/3,C67.9,Yes,No,M0,N0,T3A,Tumor Free,Yes,No,White,No,72.0,Yes,0:LIVING,94.881152,0:ALIVE OR DEAD TUMOR FREE,94.881152,0:DiseaseFree,94.881152,0:CENSORED,94.881152,EUR,BLCA


In [2]:
# quick availability audit — put this table in your report
outcome_cols = ["PFS_STATUS","PFS_MONTHS","OS_STATUS","OS_MONTHS","DFS_STATUS","DFS_MONTHS"]
audit = (clin.groupby("STUDY")[ [c for c in outcome_cols if c in clin.columns] ]
              .apply(lambda g: g.notna().mean().round(2)))
print(audit)

          PFS_STATUS  PFS_MONTHS  OS_STATUS  OS_MONTHS  DFS_STATUS  DFS_MONTHS
STUDY                                                                         
BLCA            1.00        1.00       1.00       1.00        0.46        0.46
BRCA            1.00        1.00       1.00       1.00        0.87        0.87
CESC            1.00        1.00       1.00       1.00        0.58        0.58
COADREAD        0.99        0.99       0.99       0.99        0.38        0.38
HNSC            1.00        1.00       1.00       1.00        0.25        0.25
LIHC            1.00        1.00       1.00       1.00        0.86        0.86
LUAD            0.91        0.89       0.91       0.89        0.54        0.53
LUSC            1.00        0.99       1.00       0.99        0.61        0.61
OV              0.98        0.98       0.98       0.98        0.48        0.48
STAD            1.00        0.99       1.00       0.99        0.59        0.58


In [3]:
LANDMARK = 12.0   # months

df = clin.copy()

# PFS_STATUS looks like "0:CENSORED" or "1:PROGRESSION"
df["pfs_event"] = df["PFS_STATUS"].astype(str).str.startswith("1")
df["pfs_months"] = pd.to_numeric(df["PFS_MONTHS"], errors="coerce")

funnel = {"0_all_patients": len(df)}

df = df[df["pfs_months"].notna()]
funnel["1_has_pfs_time"] = len(df)

def label_row(r):
    if r["pfs_months"] >= LANDMARK:
        return 1                      # responder
    if r["pfs_event"]:
        return 0                      # non-responder
    return np.nan                     # censored early -> drop

df["responder"] = df.apply(label_row, axis=1)
df = df[df["responder"].notna()].copy()
df["responder"] = df["responder"].astype(int)
funnel["2_labelled"] = len(df)

print(json.dumps(funnel, indent=2))
print(df["responder"].value_counts(normalize=True).round(3))
print(pd.crosstab(df["STUDY"], df["responder"]))

{
  "0_all_patients": 5359,
  "1_has_pfs_time": 5268,
  "2_labelled": 4372
}
responder
1    0.802
0    0.198
Name: proportion, dtype: float64
responder    0    1
STUDY              
BLCA       110  240
BRCA        32  884
CESC        29  209
COADREAD    76  412
HNSC       116  347
LIHC       114  187
LUAD        82  347
LUSC        62  311
OV         158  336
STAD        87  233


In [6]:
CLIN_NUM = ["AGE"]
CLIN_CAT = ["SEX", "STUDY", "AJCC_PATHOLOGIC_TUMOR_STAGE", "RACE", "SUBTYPE"]
CLIN_CAT = [c for c in CLIN_CAT if c in df.columns]

# collapse messy stage strings: "STAGE IIIA" -> "III"
if "AJCC_PATHOLOGIC_TUMOR_STAGE" in df.columns:
    df["STAGE"] = (df["AJCC_PATHOLOGIC_TUMOR_STAGE"].astype(str)
                     .str.upper().str.extract(r"STAGE\s+(IV|III|II|I)")[0])
    CLIN_CAT = [c for c in CLIN_CAT if c != "AJCC_PATHOLOGIC_TUMOR_STAGE"] + ["STAGE"]

print(df[CLIN_NUM + CLIN_CAT].isna().mean().round(3))   # missingness -> data dictionary
cohort = df[["PATIENT_ID", "STUDY", "responder"] + CLIN_NUM + CLIN_CAT].copy()
cohort.to_csv("data/processed/m1_cohort_clinical.csv", index=False)
print(cohort.shape)


AGE        0.025
SEX        0.000
STUDY      0.000
RACE       0.129
SUBTYPE    0.161
STAGE      0.194
dtype: float64
(4372, 9)


In [9]:
def load_expression(study, n_top=3000):
    """Return a samples x genes dataframe of log2 RSEM, top-variance genes only."""
    p = os.path.join(RAW, study, "data_mrna_seq_v2_rsem.txt")
    expr = pd.read_csv(p, sep="\t", low_memory=False)

    expr = expr[expr["Hugo_Symbol"].notna()]
    expr = expr.drop_duplicates(subset="Hugo_Symbol", keep="first")
    expr = expr.set_index("Hugo_Symbol").drop(columns=["Entrez_Gene_Id"], errors="ignore")

    expr = expr.apply(pd.to_numeric, errors="coerce")
    expr = np.log2(expr.clip(lower=0) + 1)              # RSEM is heavily skewed

    keep = expr.var(axis=1, skipna=True).nlargest(n_top).index
    expr = expr.loc[keep]

    return expr.T                                       # rows = samples

exprs = {s: load_expression(s) for s in STUDIES}
for s, e in exprs.items():
    print(s, e.shape)

blca_tcga_pan_can_atlas_2018 (407, 3000)
brca_tcga_pan_can_atlas_2018 (1082, 3000)
cesc_tcga_pan_can_atlas_2018 (294, 3000)
coadread_tcga_pan_can_atlas_2018 (592, 3000)
hnsc_tcga_pan_can_atlas_2018 (515, 3000)
lihc_tcga_pan_can_atlas_2018 (366, 3000)
luad_tcga_pan_can_atlas_2018 (510, 3000)
lusc_tcga_pan_can_atlas_2018 (484, 3000)
ov_tcga_pan_can_atlas_2018 (300, 3000)
stad_tcga_pan_can_atlas_2018 (412, 3000)


In [10]:
common = set.intersection(*[set(e.columns) for e in exprs.values()])
print("genes present in all studies:", len(common))

mean_var = (pd.DataFrame({s: e[list(common)].var() for s, e in exprs.items()})
              .mean(axis=1).sort_values(ascending=False))
GENES = mean_var.head(1500).index.tolist()
print("using", len(GENES), "genes")

genes present in all studies: 514
using 514 genes


In [12]:
frames = []
for s, e in exprs.items():
    x = e[GENES]
    x = x[[i.endswith("-01") for i in x.index]]         # 01 = primary tumour
    x = (x - x.mean()) / x.std(ddof=0)                  # z-score within study
    x.index = [i[:12] for i in x.index]                 # TCGA-XX-XXXX = patient
    x = x[~x.index.duplicated(keep="first")]
    frames.append(x)

expr_all = pd.concat(frames)
expr_all.index.name = "PATIENT_ID"
expr_all = expr_all.reset_index()
print(expr_all.shape)

model_tbl = cohort.merge(expr_all, on="PATIENT_ID", how="inner")
model_tbl = model_tbl.dropna(axis=1, how="all")
print("final model table:", model_tbl.shape)
print(model_tbl["responder"].value_counts())

model_tbl.to_csv("data/processed/m1_model_table.csv", index=False)

(4962, 515)
final model table: (4085, 523)
responder
1    3312
0     773
Name: count, dtype: int64


In [21]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats

FIG = "reports/figures"

# 1. class balance per cancer type
ax = pd.crosstab(model_tbl["STUDY"], model_tbl["responder"]).plot(kind="bar", stacked=True)
ax.set_ylabel("patients"); ax.set_title("Responder vs non-responder by cancer type")
plt.tight_layout(); plt.savefig(f"{FIG}/m1_class_balance.png", dpi=150); plt.close()

# 2. age distribution by outcome
plt.figure(figsize=(6,4))
sns.kdeplot(data=model_tbl, x="AGE", hue="responder", common_norm=False, fill=True)
plt.title("Age by response class")
plt.tight_layout(); plt.savefig(f"{FIG}/m1_age_by_class.png", dpi=150); plt.close()

In [14]:
print(model_tbl.columns.tolist().count("STUDY"))

2


In [15]:
model_tbl = model_tbl.loc[:, ~model_tbl.columns.duplicated()]
print(model_tbl.columns.tolist().count("STUDY"))

1


In [24]:
import os
print(os.path.exists("reports/figures/m1_class_balance.png"))
print(os.path.exists("reports/figures/m1_age_by_class.png"))

True
True


In [25]:
from statsmodels.stats.multitest import multipletests   # pip install statsmodels

resp = model_tbl[model_tbl["responder"] == 1][GENES]
nonr = model_tbl[model_tbl["responder"] == 0][GENES]

t, p = stats.ttest_ind(resp, nonr, equal_var=False, nan_policy="omit")
res = pd.DataFrame({"gene": GENES, "t": t, "p": p}).dropna()
res["q"] = multipletests(res["p"], method="fdr_bh")[1]
res = res.sort_values("p")
print(res.head(20))
print("genes with q < 0.05:", (res["q"] < 0.05).sum())
res.to_csv("reports/metrics/m1_gene_ttests.csv", index=False)

# chi-square: is stage associated with response?
if "STAGE" in model_tbl.columns:
    ct = pd.crosstab(model_tbl["STAGE"], model_tbl["responder"])
    chi2, pval, dof, _ = stats.chi2_contingency(ct)
    print(ct, f"\nchi2={chi2:.2f}  p={pval:.4g}")

         gene         t             p             q
114       IGJ  6.046410  2.004913e-09  8.915586e-07
244    POPDC3 -5.956585  3.469100e-09  8.915586e-07
164     MS4A1  5.473158  5.422526e-08  9.290595e-06
79       DKK1 -5.378325  9.176978e-08  9.614145e-06
184   PLA2G2D  5.372810  9.352282e-08  9.614145e-06
410  HLA-DQA2  5.171724  2.749188e-07  2.045811e-05
9        PIGR  5.168232  2.786124e-07  2.045811e-05
27    IGF2BP1 -5.122322  3.569364e-07  2.293317e-05
407  TNFRSF17  4.962620  7.980334e-07  4.557658e-05
311     SOX11 -4.888512  1.158147e-06  5.952874e-05
445  HLA-DQB2  4.826801  1.579064e-06  6.800064e-05
211     L1CAM -4.825636  1.587564e-06  6.800064e-05
226     CD79A  4.727808  2.547981e-06  1.007432e-04
381   POU2AF1  4.712702  2.744896e-06  1.007769e-04
507      IRF4  4.677539  3.250157e-06  1.081918e-04
100     ADAM6  4.660544  3.526058e-06  1.081918e-04
510      PLD4  4.657767  3.578328e-06  1.081918e-04
119      PAEP -4.446560  9.622952e-06  2.747887e-04
284   BMS1P2

In [26]:
import requests, time, re, xml.etree.ElementTree as ET
from sklearn.feature_extraction.text import TfidfVectorizer

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
CANCER_NAMES = {   # extend to match whatever studies you downloaded
    "BRCA": "breast carcinoma",
    "HNSC": "head and neck squamous cell carcinoma",
    "LUAD": "lung adenocarcinoma",
    "LUSC": "lung squamous cell carcinoma",
    "COADREAD": "colorectal adenocarcinoma",
    "STAD": "stomach adenocarcinoma",
    "LIHC": "liver hepatocellular carcinoma",
    "CESC": "cervical squamous cell carcinoma",
    "OV":   "ovarian serous carcinoma",
    "BLCA": "bladder urothelial carcinoma",
}

def fetch_abstracts(term, retmax=80):
    r = requests.get(f"{EUTILS}/esearch.fcgi", params={
        "db": "pubmed", "term": term, "retmax": retmax, "retmode": "json"}, timeout=30)
    ids = r.json()["esearchresult"]["idlist"]
    if not ids:
        return ""
    time.sleep(0.4)
    r2 = requests.get(f"{EUTILS}/efetch.fcgi", params={
        "db": "pubmed", "id": ",".join(ids), "retmode": "xml"}, timeout=60)
    root = ET.fromstring(r2.content)
    texts = [ "".join(a.itertext()) for a in root.iter("AbstractText") ]
    return " ".join(texts)

os.makedirs("data/interim", exist_ok=True)
corpus = {}
for code, name in CANCER_NAMES.items():
    if code not in model_tbl["STUDY"].unique():
        continue
    cache = f"data/interim/pubmed_{code}.txt"
    if os.path.exists(cache):
        corpus[code] = open(cache, encoding="utf-8").read()
    else:
        corpus[code] = fetch_abstracts(f"{name} AND chemotherapy resistance")
        open(cache, "w", encoding="utf-8").write(corpus[code])
        time.sleep(1.0)
    print(code, len(corpus[code]), "chars")

BRCA 142135 chars
HNSC 154508 chars
LUAD 153141 chars
LUSC 149123 chars
COADREAD 142466 chars
STAD 148262 chars
LIHC 137157 chars
CESC 154574 chars
OV 140837 chars
BLCA 136655 chars


In [29]:
model_tbl = model_tbl.drop(columns=["lit_resistance_score"], errors="ignore")
model_tbl = model_tbl.merge(scores, on="STUDY", how="left")
model_tbl["lit_resistance_score"] = model_tbl["lit_resistance_score"].fillna(
    model_tbl["lit_resistance_score"].mean())
model_tbl.to_csv("data/processed/m1_model_table.csv", index=False)
print(model_tbl.shape)

(4085, 525)


In [30]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.impute import SimpleImputer

Xg = SimpleImputer(strategy="median").fit_transform(model_tbl[GENES])

pca = PCA(n_components=50, random_state=42)
Xp = pca.fit_transform(Xg)
print("variance explained by 50 PCs:", pca.explained_variance_ratio_.sum().round(3))

plt.figure(figsize=(5,3.5))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("components"); plt.ylabel("cumulative variance explained")
plt.tight_layout(); plt.savefig(f"{FIG}/m1_pca_scree.png", dpi=150); plt.close()

sil = {}
for k in range(2, 9):
    lab = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Xp)
    sil[k] = silhouette_score(Xp, lab)
print(sil)
best_k = max(sil, key=sil.get); print("best k =", best_k)

model_tbl["cluster"] = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit_predict(Xp)

Xt = TSNE(n_components=2, perplexity=30, init="pca", random_state=42).fit_transform(Xp)
fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
sns.scatterplot(x=Xt[:,0], y=Xt[:,1], hue=model_tbl["STUDY"], s=12, ax=axes[0], legend="brief")
axes[0].set_title("t-SNE coloured by cancer type")
sns.scatterplot(x=Xt[:,0], y=Xt[:,1], hue=model_tbl["cluster"].astype(str), s=12, ax=axes[1])
axes[1].set_title(f"t-SNE coloured by K-Means cluster (k={best_k})")
plt.tight_layout(); plt.savefig(f"{FIG}/m1_tsne.png", dpi=150); plt.close()

print(pd.crosstab(model_tbl["cluster"], model_tbl["responder"], normalize="index").round(3))

variance explained by 50 PCs: 0.547
{2: 0.10876795704006034, 3: 0.08556160479736963, 4: 0.06200385382720736, 5: 0.06823111599190054, 6: 0.06006052637091654, 7: 0.054801901944659365, 8: 0.05267471184680067}
best k = 2
responder      0      1
cluster                
0          0.183  0.817
1          0.196  0.804


In [31]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

NUM = [c for c in ["AGE", "lit_resistance_score"] if c in model_tbl.columns]
CAT = [c for c in CLIN_CAT if c in model_tbl.columns]
y = model_tbl["responder"].values
X = model_tbl[NUM + CAT + GENES]

# stratify on label AND cancer type so no cohort lands entirely on one side
strat = model_tbl["responder"].astype(str) + "_" + model_tbl["STUDY"].astype(str)
vc = strat.value_counts()
strat = strat.where(strat.map(vc) >= 2, "rare")

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strat)
print(X_tr.shape, X_te.shape)

num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc",  StandardScaler())])
cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                     ("ohe", OneHotEncoder(handle_unknown="ignore"))])
gene_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler()),
                      ("pca", PCA(n_components=50, random_state=42))])

pre = ColumnTransformer([("num", num_pipe, NUM),
                         ("cat", cat_pipe, CAT),
                         ("gene", gene_pipe, GENES)])

(3268, 521) (817, 521)


In [32]:
MODELS = {
    "dummy":    DummyClassifier(strategy="most_frequent"),
    "logreg":   LogisticRegression(max_iter=3000, class_weight="balanced"),
    "svm_rbf":  SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42),
    "rf":       RandomForestClassifier(n_estimators=500, class_weight="balanced",
                                       random_state=42, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = ["accuracy", "precision", "recall", "f1", "roc_auc"]

rows = []
for name, est in MODELS.items():
    pipe = Pipeline([("pre", pre), ("model", est)])
    out = cross_validate(pipe, X_tr, y_tr, cv=cv, scoring=SCORING, n_jobs=-1)
    row = {"model": name}
    for m in SCORING:
        row[m] = f"{out['test_'+m].mean():.3f} ± {out['test_'+m].std():.3f}"
    rows.append(row)

results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv("reports/metrics/m1_model_comparison.csv", index=False)

  model      accuracy     precision        recall            f1       roc_auc
  dummy 0.811 ± 0.001 0.811 ± 0.001 1.000 ± 0.000 0.895 ± 0.000 0.500 ± 0.000
 logreg 0.678 ± 0.005 0.909 ± 0.013 0.670 ± 0.014 0.771 ± 0.006 0.756 ± 0.015
svm_rbf 0.707 ± 0.010 0.870 ± 0.008 0.752 ± 0.008 0.806 ± 0.007 0.701 ± 0.007
     rf 0.809 ± 0.012 0.834 ± 0.005 0.954 ± 0.011 0.890 ± 0.007 0.744 ± 0.012


In [33]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             precision_recall_curve, ConfusionMatrixDisplay)

BEST = "rf"                                   # change to whichever won
final = Pipeline([("pre", pre), ("model", MODELS[BEST])]).fit(X_tr, y_tr)

proba = final.predict_proba(X_te)[:, 1]
pred  = (proba >= 0.5).astype(int)

print(classification_report(y_te, pred, target_names=["non-responder","responder"]))
print("ROC-AUC:", round(roc_auc_score(y_te, proba), 3))
print("PR-AUC :", round(average_precision_score(y_te, proba), 3))

ConfusionMatrixDisplay(confusion_matrix(y_te, pred),
    display_labels=["non-resp","resp"]).plot(cmap="Purples")
plt.title(f"Module 1 — {BEST}, threshold 0.5")
plt.tight_layout(); plt.savefig(f"{FIG}/m1_confusion.png", dpi=150); plt.close()

               precision    recall  f1-score   support

non-responder       0.44      0.18      0.25       154
    responder       0.83      0.95      0.89       663

     accuracy                           0.80       817
    macro avg       0.63      0.56      0.57       817
 weighted avg       0.76      0.80      0.77       817

ROC-AUC: 0.735
PR-AUC : 0.921


In [34]:
from sklearn.metrics import recall_score, precision_score

sweep = []
for t in np.arange(0.2, 0.85, 0.05):
    p = (proba >= t).astype(int)
    sweep.append({"threshold": round(t,2),
                  "recall_nonresponder": recall_score(y_te, p, pos_label=0),
                  "recall_responder":    recall_score(y_te, p, pos_label=1),
                  "precision_responder": precision_score(y_te, p, pos_label=1, zero_division=0)})
sweep = pd.DataFrame(sweep)
print(sweep.to_string(index=False))

plt.figure(figsize=(6,4))
plt.plot(sweep["threshold"], sweep["recall_nonresponder"], label="recall (non-responder)")
plt.plot(sweep["threshold"], sweep["recall_responder"], label="recall (responder)")
plt.xlabel("decision threshold"); plt.ylabel("recall"); plt.legend()
plt.title("Threshold trade-off")
plt.tight_layout(); plt.savefig(f"{FIG}/m1_threshold.png", dpi=150); plt.close()

 threshold  recall_nonresponder  recall_responder  precision_responder
      0.20             0.000000          1.000000             0.811506
      0.25             0.000000          1.000000             0.811506
      0.30             0.006494          1.000000             0.812500
      0.35             0.025974          0.996983             0.815043
      0.40             0.058442          0.992459             0.819427
      0.45             0.097403          0.978884             0.823604
      0.50             0.175325          0.947210             0.831788
      0.55             0.331169          0.894419             0.852011
      0.60             0.487013          0.793363             0.869421
      0.65             0.668831          0.677225             0.898000
      0.70             0.785714          0.559578             0.918317
      0.75             0.889610          0.410256             0.941176
      0.80             0.954545          0.309201             0.966981


In [35]:
import joblib

CHOSEN_T = 0.5     # set this from the sweep above, and justify it in the report

joblib.dump({"pipeline": final, "genes": GENES, "num": NUM, "cat": CAT,
             "threshold": CHOSEN_T, "label_rule": f"PFI landmark {LANDMARK} months"},
            "models/m1_chemo_response.joblib")

metrics = {
    "module": "M1 chemotherapy response",
    "label_definition": f"responder = progression-free interval >= {LANDMARK} months",
    "studies": sorted(model_tbl["STUDY"].unique().tolist()),
    "n_patients": int(len(model_tbl)),
    "n_genes": len(GENES),
    "class_balance": model_tbl["responder"].value_counts(normalize=True).round(3).to_dict(),
    "cohort_funnel": funnel,
    "best_model": BEST,
    "threshold": CHOSEN_T,
    "test_roc_auc": round(float(roc_auc_score(y_te, proba)), 3),
    "test_pr_auc": round(float(average_precision_score(y_te, proba)), 3),
    "test_report": classification_report(y_te, pred, output_dict=True),
}
json.dump(metrics, open("reports/metrics/m1.json", "w"), indent=2)
print(json.dumps({k: metrics[k] for k in
                  ["n_patients","n_genes","best_model","test_roc_auc"]}, indent=2))

{
  "n_patients": 4085,
  "n_genes": 514,
  "best_model": "rf",
  "test_roc_auc": 0.735
}


In [36]:
with open("docs/experiment_log.md", "w") as f:
    f.write("Module 1: 10 TCGA PanCancer studies (BRCA,HNSC,LUAD,LUSC,COADREAD,STAD,LIHC,CESC,OV,BLCA)\n")
    f.write("Cohort: 4085 patients after merge, class balance ~81/19\n")
    f.write("Best model: Random Forest, test ROC-AUC 0.735\n")
    f.write("Next: try more cancer types, tune hyperparameters\n")

In [37]:
with open("docs/data_dictionary.md", "w") as f:
    f.write("""# Data Dictionary - Module 1

Source: cBioPortal PanCancer Atlas 2018, downloaded from cbioportal.org/datasets
Access date: September 2026

Studies used: BRCA, HNSC, LUAD, LUSC, COADREAD, STAD, LIHC, CESC, OV, BLCA

Key columns:
- PATIENT_ID: unique patient identifier
- AGE: age at diagnosis
- SEX: patient sex
- STAGE: cancer stage (I-IV), derived from AJCC_PATHOLOGIC_TUMOR_STAGE
- responder: 1 if progression-free >= 12 months, 0 if progressed before 12 months
- lit_resistance_score: TF-IDF score from PubMed abstracts on chemotherapy resistance per cancer type
- Gene columns: top 1500 most variable genes, z-scored within each study
""")

In [38]:
# CHOSEN_T = 0.5 selected as balanced default; threshold sweep in Chunk 9 available for adjustment if recall on non-responders needs prioritizing

In [39]:
git add -A
git commit -m "Add data dictionary, Module 1 complete"
git push

SyntaxError: invalid syntax (498342913.py, line 1)